# 7. Build Harmonic Document-Term Counts

This notebook builds the missing corpus-linguistic artifact: a broad document-by-harmonic-`n`-gram count table.

Earlier notebooks build global vocabularies and lightweight trend tables. Those trend tables intentionally track a small fixed set of globally common targets, which is useful for first-pass trend inspection but not enough for document frequency, TF-IDF, or stop-gram discovery.

This notebook creates support-thresholded `H_n` counts by grouped document type:

- `main_genre`
- `decade_1950_plus`

The output is stored in DuckDB as `harmonic_document_counts`, `harmonic_document_totals`, `harmonic_document_songs`, `harmonic_document_vocabulary`, and the analysis-ready view `harmonic_document_terms`.

## Design Choice

We still do not build a full song-by-`H_n` matrix. That would be the most flexible representation, but it is much larger and not needed yet.

Instead, this notebook counts a broader support-thresholded harmonic vocabulary across a small number of grouped documents. The default threshold keeps harmonic classes with global count at least `1,000`. This is broad enough for document-frequency analysis while keeping the exact-ID membership map manageable.

This is different from notebook 4:

- notebook 4: top global targets only, optimized for trend inspection;
- notebook 7: support-thresholded harmonic vocabulary, optimized for corpus-linguistic document-term analysis.

In [1]:
from pathlib import Path
import importlib
import os
import sys

CWD = Path.cwd()
ROOT = CWD.parent if (CWD / "utils").exists() else CWD
MPLCONFIGDIR = ROOT / ".matplotlib-cache"
MPLCONFIGDIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import duckdb
import pandas as pd

NOTEBOOK_DIR = ROOT / "notebooks"

# Force this repo's notebook utilities to win over any stale or external `utils` package.
sys.path = [p for p in sys.path if p != str(NOTEBOOK_DIR)]
sys.path.insert(0, str(NOTEBOOK_DIR))
for module_name in list(sys.modules):
    if module_name == "utils" or module_name.startswith("utils."):
        del sys.modules[module_name]

from utils import duckdb_store as ds
from utils import trend_analysis as ta

ds = importlib.reload(ds)
ta = importlib.reload(ta)

expected_files = {
    "duckdb_store": (NOTEBOOK_DIR / "utils" / "duckdb_store.py").resolve(),
    "trend_analysis": (NOTEBOOK_DIR / "utils" / "trend_analysis.py").resolve(),
}
loaded_files = {
    "duckdb_store": Path(ds.__file__).resolve(),
    "trend_analysis": Path(ta.__file__).resolve(),
}
assert loaded_files == expected_files, f"Imported wrong utility module(s): {loaded_files}; expected {expected_files}"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

RAW_PATH = ROOT / "data" / "raw" / "chordonomicon_v2.csv"
DB_PATH = ROOT / "data" / "processed" / "harmonic_trends.duckdb"

NS = tuple(range(3, 9))
MIN_GLOBAL_HARMONIC_COUNT = 1_000
MIN_DECADE = 1950
CHUNKSIZE = 25_000
MAX_ROWS = None
SHOW_PROGRESS = True
OVERWRITE = True

assert RAW_PATH.exists(), RAW_PATH
assert DB_PATH.exists(), DB_PATH

{
    "python": sys.executable,
    "duckdb_version": duckdb.__version__,
    **{name: str(path) for name, path in loaded_files.items()},
    "root": str(ROOT),
}

{'python': '/usr/local/bin/python3',
 'duckdb_version': '1.5.2',
 'duckdb_store': '/Users/juansalinas/Documents/GitHub/harmonic-trends/notebooks/utils/duckdb_store.py',
 'trend_analysis': '/Users/juansalinas/Documents/GitHub/harmonic-trends/notebooks/utils/trend_analysis.py',
 'root': '/Users/juansalinas/Documents/GitHub/harmonic-trends'}

## Preflight

Before streaming the raw songs, inspect how large the support-thresholded vocabulary will be. Lower thresholds give better coverage but require larger exact-ID membership maps and slower counting.

In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)
ds.configure_connection(con)

try:
    threshold_audit = []
    for threshold in [100, 250, 500, 1_000, 2_500, 5_000, 10_000]:
        h = con.execute(
            """
            SELECT COUNT(*) AS harmonic_classes
            FROM harmonic_ngrams
            WHERE count >= ?
            """,
            [threshold],
        ).fetchone()[0]
        exact = con.execute(
            """
            SELECT COUNT(*) AS exact_members
            FROM exact_ngrams v
            JOIN harmonic_ngrams h USING (harmonic_id)
            WHERE h.count >= ?
            """,
            [threshold],
        ).fetchone()[0]
        threshold_audit.append(
            {
                "min_global_harmonic_count": threshold,
                "harmonic_classes": h,
                "exact_members": exact,
            }
        )
finally:
    con.close()

threshold_audit = pd.DataFrame(threshold_audit)
threshold_audit

,min_global_harmonic_count,harmonic_classes,exact_members
0,100,163495,4117067
1,250,72647,2633650
2,500,38944,1845727
3,1000,21180,1288011
4,2500,9342,781388
5,5000,4878,517799
6,10000,2644,346871


## Build Document-Term Tables

This is the main streaming step. It scans the raw chord strings, maps exact `n`-grams into the selected harmonic classes, and writes the grouped document counts into DuckDB.

Progress is shown during support-thresholded target selection, raw-song scanning, and DuckDB table writes. For a quick smoke test, set `MAX_ROWS = 50_000`. For the real artifact, keep `MAX_ROWS = None`.

In [3]:
build_summary = ta.build_harmonic_document_counts(
    db_path=DB_PATH,
    raw_path=RAW_PATH,
    ns=NS,
    min_global_harmonic_count=MIN_GLOBAL_HARMONIC_COUNT,
    min_decade=MIN_DECADE,
    include_main_genre=True,
    include_decade_1950_plus=True,
    chunksize=CHUNKSIZE,
    max_rows=MAX_ROWS,
    show_progress=SHOW_PROGRESS,
    overwrite=OVERWRITE,
)

build_summary

Selecting support-thresholded harmonic targets from DuckDB (min_global_harmonic_count=1,000)...


Counting harmonic document terms: 28chunk [07:08, 15.29s/chunk, rows=679807, H_hits=2.14e+8]
Scanning songs: 100%|██████████| 679807/679807 [07:08<00:00, 1587.97song/s]


Writing harmonic document-term tables to DuckDB (400,236 count rows, 21,180 vocabulary rows)...
Harmonic document-term tables complete.


{'harmonic_document_count_rows': 400236,
 'harmonic_document_total_rows': 120,
 'harmonic_document_song_rows': 20,
 'harmonic_document_vocabulary_rows': 21180,
 'min_global_harmonic_count': 1000}

## Validate Outputs

Counts are normalized by all windows in the document, not only by selected-vocabulary hits. This keeps frequencies comparable across thresholds.

In [4]:
con = duckdb.connect(str(DB_PATH), read_only=True)
ds.configure_connection(con)

try:
    table_counts = con.execute(
        """
        SELECT 'harmonic_document_counts' AS table_name, COUNT(*) AS row_count FROM harmonic_document_counts
        UNION ALL SELECT 'harmonic_document_totals', COUNT(*) FROM harmonic_document_totals
        UNION ALL SELECT 'harmonic_document_songs', COUNT(*) FROM harmonic_document_songs
        UNION ALL SELECT 'harmonic_document_vocabulary', COUNT(*) FROM harmonic_document_vocabulary
        ORDER BY table_name
        """
    ).fetchdf()

    document_summary = con.execute(
        """
        WITH observed AS (
            SELECT
                document_type,
                n,
                COUNT(DISTINCT harmonic_id) AS observed_harmonic_classes
            FROM harmonic_document_counts
            GROUP BY document_type, n
        )
        SELECT
            t.document_type,
            t.n,
            COUNT(DISTINCT t.document_value) AS n_documents,
            SUM(t.total_windows)::BIGINT AS total_windows,
            MIN(t.total_windows) AS min_windows,
            MEDIAN(t.total_windows) AS median_windows,
            MAX(t.total_windows) AS max_windows,
            o.observed_harmonic_classes
        FROM harmonic_document_totals t
        LEFT JOIN observed o USING (document_type, n)
        GROUP BY t.document_type, t.n, o.observed_harmonic_classes
        ORDER BY t.document_type, t.n
        """
    ).fetchdf()

    sample_terms = con.execute(
        """
        SELECT *
        FROM harmonic_document_terms
        ORDER BY document_type, document_value, n, count DESC, harmonic_id
        LIMIT 25
        """
    ).fetchdf()
finally:
    con.close()

table_counts, document_summary, sample_terms

(                     table_name  row_count
 0      harmonic_document_counts     400236
 1       harmonic_document_songs         20
 2      harmonic_document_totals        120
 3  harmonic_document_vocabulary      21180,
        document_type  n  n_documents  total_windows  min_windows  median_windows  max_windows  observed_harmonic_classes
 0   decade_1950_plus  3            8       31224962        95923       2279545.0     14216707                       2239
 1   decade_1950_plus  4            8       30803236        94175       2248616.0     14031272                       3428
 2   decade_1950_plus  5            8       30381600        92427       2217697.0     13845874                       4275
 3   decade_1950_plus  6            8       29960122        90679       2186786.0     13660551                       4283
 4   decade_1950_plus  7            8       29539070        88933       2155925.0     13475344                       3693
 5   decade_1950_plus  8            8       291

## Handoff

Notebook 8 consumes `harmonic_document_terms` for document frequency, stop-gram candidates, TF-IDF, entropy, and one-vs-rest enrichment.

If notebook 8 reports that `idf = 1` for everything again, then the support threshold is still too restrictive or the document groups are too coarse. Lower `MIN_GLOBAL_HARMONIC_COUNT` here, rerun this notebook, and then rerun notebook 8.